In this demonstration file, I will explore techniques for enhancing interpretability in medical imaging using pre-trained models from TorchXRayVision (https://github.com/mlmed/torchxrayvision).

# Setup

In [ ]:
%pip install torchxrayvision captum gradio
from pathlib import Path
import subprocess
if not Path('gifsplanation').is_dir():
    subprocess.check_call(['git', 'clone', 'https://github.com/mlmed/gifsplanation'])

from pathlib import Path
from typing import Callable

import gradio as gr
import numpy as np
import torchxrayvision as xrv
import torch
import torchvision
import skimage
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import sys
import os
sys.path.insert(0,"./gifsplanation/")

from captum.attr import IntegratedGradients, Saliency, InputXGradient
import attribution

from urllib.request import urlretrieve
urlretrieve('https://raw.githubusercontent.com/mlmed/torchxrayvision/master/tests/16747_3_1.jpg', '16747_3_1.jpg')

In [ ]:
def make_fig(plot_matrix):
    fig = plt.figure()
    plt.imshow(plot_matrix, cmap=plt.cm.hot)
    #plt.title(plot_title)
    ax = plt.gca()
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    return fig

In [ ]:
def xrv_prepare_image(image):
    img = xrv.datasets.normalize(image, 255) # convert 8-bit image to [-1024, 1024] range
    img = img.mean(2)[None, ...] if img.ndim > 2 else img[None, ...] # Make single color channel
    transform = torchvision.transforms.Compose([xrv.datasets.XRayCenterCrop(),xrv.datasets.XRayResizer(224)])
    img = transform(img)
    img = torch.from_numpy(img)
    
    return img[None,...]

In [ ]:
def predict(image, model_choice):
    """Function that serves predictions."""
    img = xrv_prepare_image(image)
    model = xrv.models.DenseNet(weights=model_choice)
    model.eval()

    outputs = model(img)
    scores =  outputs[0].detach().cpu().numpy().astype(float) #conversion to np.float is needed for visualization with gr.Label
    label = dict(zip(model.pathologies, scores) )
    return label

In [ ]:
def explain_gradient(image, model_choice, target):
    """Function that serves explanations."""
    
    model = xrv.models.DenseNet(weights=model_choice)
    model.eval()
    input = xrv_prepare_image(image)
    
    #Saliency
    saliency = Saliency(model)
    attr = saliency.attribute(input, target=model.pathologies.index(target))
    fig1 = make_fig( np.abs(attr[0,0].numpy()) )

    return fig1

def explain_input_x_gradient(image, model_choice, target):
    """Function that serves explanations."""
    
    model = xrv.models.DenseNet(weights=model_choice)
    model.eval()
    input = xrv_prepare_image(image)
    
    #InputXGradient
    ixg = InputXGradient(model)
    attr = ixg.attribute(input, target=model.pathologies.index(target))
    fig2 = make_fig( np.abs(attr[0,0].detach().cpu().numpy()) ) 
    return fig2
    
def explain_integrated_gradients(image, model_choice, target):
    """Function that serves explanations."""
    
    model = xrv.models.DenseNet(weights=model_choice)
    model.eval()
    input = xrv_prepare_image(image)
    
    #IntegratedGradients
    ig = IntegratedGradients(model)
    attr = ig.attribute(input, target=model.pathologies.index(target))
    fig3 = make_fig( np.abs(attr[0,0].detach().cpu().numpy()) )
    return fig3

def explain_gifsplanation(image, model_choice, target):
    """Function that serves explanations."""
    
    model = xrv.models.DenseNet(weights=model_choice)
    model.eval()
    input = xrv_prepare_image(image)
    
    #Gifsplanation
    input.requires_grad=False
    ae = xrv.autoencoders.ResNetAE(weights="101-elastic")
    movie = attribution.generate_video(input, model, target, ae, target_filename="test", border=False, show=False,
                        ffmpeg_path="ffmpeg")

    return movie

In [ ]:
model_choices = ["densenet121-res224-all", "densenet121-res224-rsna","densenet121-res224-nih","densenet121-res224-pc","densenet121-res224-chex","densenet121-res224-mimic_nb","densenet121-res224-mimic_ch"]

target_choices = ['Atelectasis','Consolidation','Infiltration','Pneumothorax','Edema','Emphysema',
                'Fibrosis','Effusion','Pneumonia','Pleural_Thickening','Cardiomegaly','Nodule','Mass',
                'Hernia','Lung Lesion','Fracture','Lung Opacity','Enlarged Cardiomediastinum']

In [ ]:
# build a basic browser interface to a Python function
frontend = gr.Blocks()

with frontend:
  gr.Markdown(
    """
    # XRayMind: Prognostics for Chest Radiography via Transfer Learning
    Human-in-the-loop interface for exploring TorchXRayVision models and data.
    This automated tool allows users to diagnose chest radiographs by selecting their choice of model and explaining the model's predictions.
    Users can simply upload an image of a chest x-ray into the algorithm to create a report.
    In the example, there are 3 chest x-ray images available from the NIH dataset that can be used to test the algorithm.
    """
    )
  #layout
  with gr.Tab("Prediction"):
    with gr.Row():
      with gr.Column():
        input_image = gr.Image(label="X-ray image")
        select_model = gr.Dropdown(label="Select model", choices=model_choices, value=model_choices[0])
        with gr.Row():
          submit_button = gr.Button("Submit")
        gr.Examples(["16747_3_1.jpg"], inputs=input_image)

      with gr.Column():
        label = gr.Label(label="Multiclass predictions")

  submit_button.click(predict, [input_image, select_model], label) 

  with gr.Tab("Explanation"):

    with gr.Row():
      with gr.Column():
        select_target = gr.Dropdown(label="Select target", choices=target_choices)
        with gr.Row(equal_height=True):
          with gr.Column():
            with gr.Tab("Im..."):
              original_image = gr.Image(label='Original', interactive=False)
          with gr.Column():
            with gr.Tab("Gra..."):
              gradient_plot = gr.Plot(label="Gradient")
            with gr.Tab("XGr..."):
              input_x_gradient_plot = gr.Plot(label='InputXGradient')
            with gr.Tab("Int..."):
              integrated_gradients_plot = gr.Plot(label="IntegratedGradients")
            with gr.Tab("Gif..."):
              gifsplanation_vid = gr.Video(label="Gifsplanation")
          
  input_image.change(lambda s: s, inputs=input_image, outputs=original_image)
  select_target.change(explain_gradient, inputs=[input_image,select_model, select_target], outputs=gradient_plot)
  select_target.change(explain_input_x_gradient, inputs=[input_image,select_model, select_target], outputs=input_x_gradient_plot)
  select_target.change(explain_integrated_gradients, inputs=[input_image,select_model, select_target], outputs=integrated_gradients_plot)
  select_target.change(explain_gifsplanation, inputs=[input_image,select_model, select_target], outputs=gifsplanation_vid)

frontend.launch(share=True, show_error=True)

In [ ]:
frontend.close()